# SLA Violation Prediction Platform - Real-time Simulation

This notebook demonstrates the real-time capabilities of the SLA Violation Prediction and Anomaly Detection platform.

## Features Demonstrated:
- Real-time telemetry data generation
- SLA violation prediction using trained ML models
- Anomaly detection with Isolation Forest
- Interactive visualization of predictions over time
- API integration with the FastAPI backend
- Alert simulation and monitoring

## Prerequisites:
- Backend API running on http://localhost:8000
- Required Python packages installed
- Network connectivity to the API

In [ ]:
# Import required libraries
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
from datetime import datetime, timedelta
import asyncio
import websockets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configuration
API_BASE_URL = "http://localhost:8000"
WS_URL = "ws://localhost:8000/ws/telemetry"

print("Libraries imported successfully!")
print(f"API Base URL: {API_BASE_URL}")
print(f"WebSocket URL: {WS_URL}")

## 1. API Health Check and Model Status

In [ ]:
# Check API health
def check_api_health():
    try:
        response = requests.get(f"{API_BASE_URL}/health")
        if response.status_code == 200:
            health_data = response.json()
            print("✅ API Health Check Passed")
            print(f"Status: {health_data['status']}")
            print(f"Timestamp: {health_data['timestamp']}")
            print(f"Version: {health_data['version']}")
            return True
        else:
            print(f"❌ API Health Check Failed: {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ API Connection Error: {str(e)}")
        return False

# Check model metrics
def check_model_metrics():
    try:
        response = requests.get(f"{API_BASE_URL}/model/metrics")
        if response.status_code == 200:
            metrics = response.json()
            print("\n📊 Model Metrics:")
            print(f"Model: {metrics['model_name']}")
            print(f"Version: {metrics['model_version']}")
            print(f"Accuracy: {metrics['accuracy']:.4f}")
            print(f"AUC Score: {metrics['auc_score']:.4f}")
            print(f"Training Data Size: {metrics['training_data_size']}")
            return metrics
        else:
            print(f"❌ Model Metrics Error: {response.status_code}")
            return None
    except Exception as e:
        print(f"❌ Model Metrics Error: {str(e)}")
        return None

# Run health checks
api_healthy = check_api_health()
model_metrics = check_model_metrics()

if not api_healthy:
    print("\n⚠️ Please ensure the backend API is running on http://localhost:8000")
    print("Run: cd backend && uvicorn app.main:app --reload")

## 2. Synthetic Telemetry Data Generator

In [ ]:
class TelemetryGenerator:
    """Generate realistic network telemetry data with configurable patterns."""
    
    def __init__(self):
        self.base_params = {
            'bandwidth': 2.0,
            'throughput': 1.8,
            'congestion': 0.1,
            'packet_loss': 0.5,
            'latency': 8.0,
            'jitter': 0.2,
            'percentage_video_occupancy': 45.0,
            'bitrate_video': 500.0,
            'number_videos': 2
        }
        self.anomaly_probability = 0.1
        self.burst_probability = 0.05
        self.time_step = 0
    
    def generate_normal_sample(self):
        """Generate a normal telemetry sample."""
        sample = {}
        
        # Add time-based variations (daily patterns)
        hour_factor = np.sin(self.time_step * 2 * np.pi / 24) * 0.3 + 1.0
        
        for param, base_value in self.base_params.items():
            if param in ['bandwidth', 'throughput']:
                # Higher during peak hours
                sample[param] = base_value * hour_factor * np.random.normal(1.0, 0.1)
            elif param in ['congestion', 'packet_loss', 'latency', 'jitter']:
                # Higher during peak hours (inverse relationship)
                sample[param] = base_value * (2.0 - hour_factor) * np.random.normal(1.0, 0.2)
            else:
                sample[param] = base_value * np.random.normal(1.0, 0.15)
        
        # Ensure realistic bounds
        sample['bandwidth'] = max(0.1, min(10.0, sample['bandwidth']))
        sample['throughput'] = max(0.1, min(sample['bandwidth'], sample['throughput']))
        sample['congestion'] = max(0.0, min(1.0, sample['congestion']))
        sample['packet_loss'] = max(0.0, min(10.0, sample['packet_loss']))
        sample['latency'] = max(1.0, min(100.0, sample['latency']))
        sample['jitter'] = max(0.0, min(5.0, sample['jitter']))
        sample['percentage_video_occupancy'] = max(0.0, min(100.0, sample['percentage_video_occupancy']))
        sample['bitrate_video'] = max(0.0, min(2000.0, sample['bitrate_video']))
        sample['number_videos'] = max(0, min(10, int(sample['number_videos'])))
        
        return sample
    
    def generate_anomaly_sample(self):
        """Generate an anomalous telemetry sample."""
        sample = self.generate_normal_sample()
        
        # Introduce anomalies
        anomaly_type = np.random.choice(['high_latency', 'packet_loss', 'congestion', 'throughput_drop'])
        
        if anomaly_type == 'high_latency':
            sample['latency'] *= np.random.uniform(3.0, 8.0)
            sample['jitter'] *= np.random.uniform(2.0, 5.0)
        elif anomaly_type == 'packet_loss':
            sample['packet_loss'] *= np.random.uniform(5.0, 15.0)
            sample['throughput'] *= np.random.uniform(0.3, 0.7)
        elif anomaly_type == 'congestion':
            sample['congestion'] *= np.random.uniform(3.0, 8.0)
            sample['latency'] *= np.random.uniform(2.0, 4.0)
        elif anomaly_type == 'throughput_drop':
            sample['throughput'] *= np.random.uniform(0.1, 0.4)
            sample['bandwidth'] *= np.random.uniform(0.2, 0.6)
        
        return sample
    
    def generate_burst_sample(self):
        """Generate a burst/spike sample."""
        sample = self.generate_normal_sample()
        
        # Create traffic burst
        sample['number_videos'] *= np.random.uniform(2.0, 5.0)
        sample['percentage_video_occupancy'] *= np.random.uniform(1.5, 2.5)
        sample['bitrate_video'] *= np.random.uniform(1.5, 3.0)
        sample['congestion'] *= np.random.uniform(2.0, 4.0)
        
        return sample
    
    def generate_sample(self):
        """Generate a telemetry sample based on probabilities."""
        self.time_step += 1
        
        rand = np.random.random()
        
        if rand < self.burst_probability:
            return self.generate_burst_sample(), 'burst'
        elif rand < self.burst_probability + self.anomaly_probability:
            return self.generate_anomaly_sample(), 'anomaly'
        else:
            return self.generate_normal_sample(), 'normal'

# Test the generator
generator = TelemetryGenerator()
sample, sample_type = generator.generate_sample()

print("📡 Telemetry Generator Initialized")
print(f"Sample Type: {sample_type}")
print("Sample Data:")
for key, value in sample.items():
    print(f"  {key}: {value:.3f}")

## 3. Real-time Prediction and Monitoring

In [ ]:
class RealTimeMonitor:
    """Real-time monitoring and prediction system."""
    
    def __init__(self, api_base_url):
        self.api_base_url = api_base_url
        self.generator = TelemetryGenerator()
        self.history = []
        self.predictions = []
        self.anomalies = []
        self.alerts = []
    
    def send_telemetry(self, data):
        """Send telemetry data to API and get prediction."""
        try:
            # Send prediction request
            response = requests.post(f"{self.api_base_url}/predict-and-store/", json=data)
            if response.status_code == 200:
                result = response.json()
                return result
            else:
                print(f"API Error: {response.status_code}")
                return None
        except Exception as e:
            print(f"Request Error: {str(e)}")
            return None
    
    def check_anomaly(self, data):
        """Check for anomalies in the data."""
        try:
            response = requests.post(f"{self.api_base_url}/anomaly/", json=data)
            if response.status_code == 200:
                result = response.json()
                return result
            else:
                return None
        except Exception as e:
            return None
    
    def run_simulation(self, duration_minutes=5, interval_seconds=2):
        """Run real-time simulation."""
        print(f"🚀 Starting {duration_minutes}-minute simulation...")
        print(f"📊 Sending data every {interval_seconds} seconds")
        
        start_time = time.time()
        end_time = start_time + (duration_minutes * 60)
        
        iteration = 0
        
        while time.time() < end_time:
            iteration += 1
            current_time = datetime.now()
            
            # Generate telemetry data
            telemetry_data, data_type = self.generator.generate_sample()
            
            # Send to API for prediction
            prediction_result = self.send_telemetry(telemetry_data)
            
            # Check for anomalies
            anomaly_result = self.check_anomaly(telemetry_data)
            
            # Store results
            record = {
                'timestamp': current_time,
                'iteration': iteration,
                'data_type': data_type,
                'telemetry': telemetry_data,
                'prediction': prediction_result,
                'anomaly': anomaly_result
            }
            
            self.history.append(record)
            
            # Check for SLA violations and anomalies
            if prediction_result and prediction_result.get('sla_violation') == 1:
                self.predictions.append(record)
                risk_score = prediction_result.get('sla_risk_score', 0)
                if risk_score > 0.8:
                    alert = {
                        'timestamp': current_time,
                        'type': 'SLA_VIOLATION',
                        'risk_score': risk_score,
                        'data': telemetry_data
                    }
                    self.alerts.append(alert)
                    print(f"🚨 HIGH RISK SLA VIOLATION DETECTED! Risk Score: {risk_score:.3f}")
            
            if anomaly_result and anomaly_result.get('anomaly_flag'):
                self.anomalies.append(record)
                anomaly_score = anomaly_result.get('anomaly_score', 0)
                print(f"⚠️ ANOMALY DETECTED! Score: {anomaly_score:.3f}")
            
            # Progress update
            if iteration % 10 == 0:
                elapsed = time.time() - start_time
                remaining = (duration_minutes * 60) - elapsed
                print(f"📈 Iteration {iteration} | {remaining/60:.1f} min remaining | "
                      f"Violations: {len(self.predictions)} | Anomalies: {len(self.anomalies)}")
            
            time.sleep(interval_seconds)
        
        print(f"\n✅ Simulation completed!")
        print(f"📊 Total samples: {len(self.history)}")
        print(f"🚨 SLA violations: {len(self.predictions)}")
        print(f"⚠️ Anomalies detected: {len(self.anomalies)}")
        print(f"🔔 High-risk alerts: {len(self.alerts)}")
    
    def get_summary_stats(self):
        """Get summary statistics from the simulation."""
        if not self.history:
            return None
        
        df = pd.DataFrame([{
            'timestamp': record['timestamp'],
            'latency': record['telemetry']['latency'],
            'packet_loss': record['telemetry']['packet_loss'],
            'throughput': record['telemetry']['throughput'],
            'congestion': record['telemetry']['congestion'],
            'sla_violation': record['prediction']['sla_violation'] if record['prediction'] else 0,
            'sla_risk_score': record['prediction']['sla_risk_score'] if record['prediction'] else 0,
            'anomaly_flag': record['anomaly']['anomaly_flag'] if record['anomaly'] else False,
            'data_type': record['data_type']
        } for record in self.history])
        
        return df

# Initialize monitor
monitor = RealTimeMonitor(API_BASE_URL)
print("🔍 Real-time Monitor initialized")

## 4. Run Real-time Simulation

In [ ]:
# Run the simulation (adjust duration as needed)
if api_healthy:
    print("Starting real-time simulation...")
    print("This will generate synthetic telemetry data and send it to the API for prediction.")
    print("Watch for SLA violations and anomaly alerts!\n")
    
    # Run simulation for 3 minutes with 2-second intervals
    monitor.run_simulation(duration_minutes=3, interval_seconds=2)
else:
    print("⚠️ Skipping simulation - API not available")
    print("Please start the backend API and re-run this cell")

## 5. Visualization and Analysis

In [ ]:
# Get simulation data
if monitor.history:
    df = monitor.get_summary_stats()
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('SLA Violation Prediction - Real-time Simulation Results', fontsize=16, fontweight='bold')
    
    # 1. SLA Violations over time
    ax1 = axes[0, 0]
    violation_data = df.groupby(df['timestamp'].dt.floor('30S'))['sla_violation'].sum().reset_index()
    ax1.plot(violation_data['timestamp'], violation_data['sla_violation'], 'r-', linewidth=2, marker='o')
    ax1.set_title('SLA Violations Over Time', fontweight='bold')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Violations Count')
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis='x', rotation=45)
    
    # 2. Risk Score Distribution
    ax2 = axes[0, 1]
    ax2.hist(df['sla_risk_score'], bins=20, alpha=0.7, color='orange', edgecolor='black')
    ax2.axvline(x=0.8, color='red', linestyle='--', linewidth=2, label='High Risk Threshold')
    ax2.set_title('SLA Risk Score Distribution', fontweight='bold')
    ax2.set_xlabel('Risk Score')
    ax2.set_ylabel('Frequency')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Network Metrics over time
    ax3 = axes[1, 0]
    time_data = df.set_index('timestamp')
    ax3.plot(time_data.index, time_data['latency'], label='Latency (ms)', alpha=0.8)
    ax3.plot(time_data.index, time_data['packet_loss'], label='Packet Loss (%)', alpha=0.8)
    ax3.plot(time_data.index, time_data['congestion'] * 10, label='Congestion (x10)', alpha=0.8)
    ax3.set_title('Network Metrics Over Time', fontweight='bold')
    ax3.set_xlabel('Time')
    ax3.set_ylabel('Value')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.tick_params(axis='x', rotation=45)
    
    # 4. Anomaly Detection Results
    ax4 = axes[1, 1]
    anomaly_counts = df['data_type'].value_counts()
    colors = ['lightgreen', 'orange', 'red']
    wedges, texts, autotexts = ax4.pie(anomaly_counts.values, labels=anomaly_counts.index, 
                                       autopct='%1.1f%%', colors=colors, startangle=90)
    ax4.set_title('Data Type Distribution', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\n📊 SIMULATION SUMMARY")
    print("=" * 50)
    print(f"Total Samples: {len(df)}")
    print(f"SLA Violations: {df['sla_violation'].sum()} ({df['sla_violation'].mean()*100:.1f}%)")
    print(f"Anomalies Detected: {df['anomaly_flag'].sum()} ({df['anomaly_flag'].mean()*100:.1f}%)")
    print(f"High-Risk Alerts: {len(monitor.alerts)}")
    print(f"Average Risk Score: {df['sla_risk_score'].mean():.3f}")
    print(f"Max Risk Score: {df['sla_risk_score'].max():.3f}")
    
    print("\n📈 NETWORK METRICS SUMMARY")
    print("=" * 50)
    print(f"Average Latency: {df['latency'].mean():.2f} ms")
    print(f"Average Packet Loss: {df['packet_loss'].mean():.2f}%")
    print(f"Average Throughput: {df['throughput'].mean():.2f} Mbps")
    print(f"Average Congestion: {df['congestion'].mean():.3f}")
    
    # Data type breakdown
    print("\n🔍 DATA TYPE BREAKDOWN")
    print("=" * 50)
    for data_type, count in df['data_type'].value_counts().items():
        percentage = (count / len(df)) * 100
        print(f"{data_type.capitalize()}: {count} samples ({percentage:.1f}%)")
        
else:
    print("No simulation data available. Please run the simulation first.")

## 6. Model Explanation and Feature Importance

In [ ]:
# Get model explanation for a high-risk sample
if monitor.predictions:
    # Find a high-risk prediction
    high_risk_sample = None
    for record in monitor.history:
        if (record['prediction'] and 
            record['prediction'].get('sla_risk_score', 0) > 0.7):
            high_risk_sample = record
            break
    
    if high_risk_sample:
        # Get the telemetry ID from the database
        try:
            # Get recent telemetry records
            response = requests.get(f"{API_BASE_URL}/telemetry/?limit=10")
            if response.status_code == 200:
                telemetry_records = response.json()
                if telemetry_records:
                    # Use the most recent record for explanation
                    latest_record = telemetry_records[0]
                    record_id = latest_record['id']
                    
                    # Get SHAP explanation
                    explanation_response = requests.get(f"{API_BASE_URL}/explain/{record_id}")
                    if explanation_response.status_code == 200:
                        explanation = explanation_response.json()
                        
                        print("🔍 MODEL EXPLANATION (SHAP Values)")
                        print("=" * 50)
                        print(f"Record ID: {record_id}")
                        print(f"Prediction: {'SLA Violation' if explanation['prediction'] else 'No Violation'}")
                        print(f"Risk Score: {explanation['risk_score']:.3f}")
                        
                        print("\n📊 Feature Contributions (SHAP Values):")
                        shap_values = explanation['shap_values']
                        
                        # Sort features by absolute SHAP value
                        sorted_features = sorted(shap_values.items(), 
                                               key=lambda x: abs(x[1]), reverse=True)
                        
                        for feature, value in sorted_features:
                            direction = "↑" if value > 0 else "↓"
                            print(f"  {feature}: {value:+.4f} {direction}")
                        
                        # Visualize SHAP values
                        features = list(shap_values.keys())
                        values = list(shap_values.values())
                        
                        plt.figure(figsize=(10, 6))
                        colors = ['red' if v > 0 else 'blue' for v in values]
                        bars = plt.barh(features, values, color=colors, alpha=0.7)
                        plt.xlabel('SHAP Value (Impact on Prediction)')
                        plt.title('Feature Importance for SLA Violation Prediction\n(Red: Increases Risk, Blue: Decreases Risk)')
                        plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
                        plt.grid(True, alpha=0.3)
                        
                        # Add value labels on bars
                        for bar, value in zip(bars, values):
                            plt.text(value + (0.001 if value >= 0 else -0.001), 
                                   bar.get_y() + bar.get_height()/2, 
                                   f'{value:.3f}', 
                                   ha='left' if value >= 0 else 'right', 
                                   va='center', fontsize=9)
                        
                        plt.tight_layout()
                        plt.show()
                        
                    else:
                        print(f"❌ Could not get explanation: {explanation_response.status_code}")
                else:
                    print("No telemetry records found in database")
            else:
                print(f"❌ Could not fetch telemetry records: {response.status_code}")
        except Exception as e:
            print(f"❌ Error getting model explanation: {str(e)}")
    else:
        print("No high-risk samples found in this simulation")
else:
    print("No predictions available for explanation")

## 7. Alert System Testing

In [ ]:
# Test alert configuration
def test_alert_system():
    print("🔔 Testing Alert System")
    print("=" * 30)
    
    # Configure alerts
    alert_config = {
        "alert_type": "sla_violation",
        "threshold": 0.8,
        "channels": ["telegram", "email"]
    }
    
    try:
        response = requests.post(f"{API_BASE_URL}/alerts/configure", json=alert_config)
        if response.status_code == 200:
            result = response.json()
            print(f"✅ Alert configuration successful: {result['message']}")
            
            # Generate a high-risk sample to trigger alert
            print("\n🚨 Generating high-risk sample to test alerts...")
            
            # Create a sample that should trigger SLA violation
            high_risk_sample = {
                "bandwidth": 0.5,  # Low bandwidth
                "throughput": 0.3,  # Very low throughput
                "congestion": 0.8,  # High congestion
                "packet_loss": 5.0,  # High packet loss
                "latency": 50.0,  # High latency
                "jitter": 2.0,  # High jitter
                "percentage_video_occupancy": 80.0,
                "bitrate_video": 1000.0,
                "number_videos": 5
            }
            
            # Send the high-risk sample
            prediction_response = requests.post(f"{API_BASE_URL}/predict-and-store/", 
                                              json=high_risk_sample)
            
            if prediction_response.status_code == 200:
                prediction = prediction_response.json()
                print(f"📊 Prediction Result:")
                print(f"  SLA Violation: {'Yes' if prediction['sla_violation'] else 'No'}")
                print(f"  Risk Score: {prediction['sla_risk_score']:.3f}")
                
                if prediction['sla_risk_score'] > 0.8:
                    print(f"🚨 HIGH RISK DETECTED! Alert should be triggered.")
                    print(f"📧 Check your configured alert channels for notifications.")
                else:
                    print(f"ℹ️ Risk score below threshold, no alert triggered.")
            else:
                print(f"❌ Prediction failed: {prediction_response.status_code}")
                
        else:
            print(f"❌ Alert configuration failed: {response.status_code}")
            print(response.text)
            
    except Exception as e:
        print(f"❌ Alert system test error: {str(e)}")

# Run alert system test
if api_healthy:
    test_alert_system()
else:
    print("⚠️ Skipping alert test - API not available")

## 8. Performance Metrics and Model Evaluation

In [ ]:
# Evaluate model performance on simulation data
if monitor.history and len(monitor.history) > 10:
    print("📈 MODEL PERFORMANCE EVALUATION")
    print("=" * 50)
    
    # Calculate performance metrics
    df = monitor.get_summary_stats()
    
    # Confusion matrix data
    true_violations = df[df['data_type'].isin(['anomaly', 'burst'])]['sla_violation'].sum()
    predicted_violations = df['sla_violation'].sum()
    total_anomalous = len(df[df['data_type'].isin(['anomaly', 'burst'])])
    total_normal = len(df[df['data_type'] == 'normal'])
    
    print(f"📊 Prediction Statistics:")
    print(f"  Total Predictions: {len(df)}")
    print(f"  Predicted Violations: {predicted_violations}")
    print(f"  Anomalous Samples: {total_anomalous}")
    print(f"  Normal Samples: {total_normal}")
    
    # Risk score analysis
    high_risk_count = len(df[df['sla_risk_score'] > 0.8])
    medium_risk_count = len(df[(df['sla_risk_score'] > 0.5) & (df['sla_risk_score'] <= 0.8)])
    low_risk_count = len(df[df['sla_risk_score'] <= 0.5])
    
    print(f"\n🎯 Risk Score Distribution:")
    print(f"  High Risk (>0.8): {high_risk_count} ({high_risk_count/len(df)*100:.1f}%)")
    print(f"  Medium Risk (0.5-0.8): {medium_risk_count} ({medium_risk_count/len(df)*100:.1f}%)")
    print(f"  Low Risk (≤0.5): {low_risk_count} ({low_risk_count/len(df)*100:.1f}%)")
    
    # Anomaly detection performance
    anomaly_detection_rate = df['anomaly_flag'].mean()
    print(f"\n🔍 Anomaly Detection:")
    print(f"  Detection Rate: {anomaly_detection_rate*100:.1f}%")
    print(f"  Anomalies Found: {df['anomaly_flag'].sum()}")
    
    # Response time analysis (if available)
    print(f"\n⏱️ System Performance:")
    print(f"  Average Processing Time: ~{interval_seconds}s (simulated)")
    print(f"  Data Points per Minute: {60/interval_seconds:.0f}")
    print(f"  Total Simulation Duration: {len(df)*interval_seconds/60:.1f} minutes")
    
    # Create performance visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Risk score over time
    axes[0].plot(range(len(df)), df['sla_risk_score'], alpha=0.7, color='orange')
    axes[0].axhline(y=0.8, color='red', linestyle='--', label='High Risk Threshold')
    axes[0].axhline(y=0.5, color='yellow', linestyle='--', label='Medium Risk Threshold')
    axes[0].set_title('Risk Score Over Time')
    axes[0].set_xlabel('Sample Number')
    axes[0].set_ylabel('Risk Score')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Violation prediction accuracy by data type
    violation_by_type = df.groupby('data_type')['sla_violation'].agg(['count', 'sum', 'mean'])
    axes[1].bar(violation_by_type.index, violation_by_type['mean'], 
               color=['green', 'orange', 'red'], alpha=0.7)
    axes[1].set_title('SLA Violation Rate by Data Type')
    axes[1].set_xlabel('Data Type')
    axes[1].set_ylabel('Violation Rate')
    axes[1].grid(True, alpha=0.3)
    
    # Cumulative violations and anomalies
    cumulative_violations = df['sla_violation'].cumsum()
    cumulative_anomalies = df['anomaly_flag'].cumsum()
    axes[2].plot(range(len(df)), cumulative_violations, label='Cumulative Violations', color='red')
    axes[2].plot(range(len(df)), cumulative_anomalies, label='Cumulative Anomalies', color='orange')
    axes[2].set_title('Cumulative Detections')
    axes[2].set_xlabel('Sample Number')
    axes[2].set_ylabel('Count')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
else:
    print("Insufficient data for performance evaluation. Please run a longer simulation.")

## 9. Conclusion and Next Steps

This notebook has demonstrated the complete SLA Violation Prediction and Anomaly Detection platform in action:

### ✅ What We've Accomplished:
1. **Real-time Data Generation**: Synthetic network telemetry with realistic patterns
2. **ML Prediction**: Live SLA violation prediction using trained XGBoost models
3. **Anomaly Detection**: Real-time anomaly detection using Isolation Forest
4. **Alert System**: Configurable alerts for high-risk scenarios
5. **Model Explainability**: SHAP-based feature importance analysis
6. **Performance Monitoring**: Comprehensive metrics and visualizations

### 🚀 Next Steps:
1. **Scale Testing**: Run longer simulations with more data points
2. **Model Tuning**: Adjust thresholds based on business requirements
3. **Integration**: Connect to real network monitoring systems
4. **Dashboard**: Use the React frontend for real-time monitoring
5. **Deployment**: Deploy to production using the provided Docker configurations

### 📊 Platform Features:
- **Backend API**: FastAPI with comprehensive endpoints
- **Frontend Dashboard**: React with interactive maps and real-time updates
- **Machine Learning**: XGBoost, Random Forest, and Isolation Forest models
- **Real-time Processing**: WebSocket support for live updates
- **Monitoring**: Grafana dashboards and Prometheus metrics
- **Deployment**: Docker containers with cloud deployment scripts

The platform is ready for production use and can be easily extended with additional features and integrations.